# 🔎 UFVAI — Agente de IA para Pesquisa Científica

O **UFVAI** é um assistente de IA no Google Colab, sobre a arquitetura **OpenCode**, que automatiza o fluxo acadêmico — do levantamento bibliográfico à estruturação de artigos — com dados públicos do Brasil (**IBGE**, **DataSUS**, **INPE**, **CONAB**, **Banco Central** e outras).

| Benefício | Detalhe |
|-----------|---------|
| **100% Gratuito** | Infraestrutura em nuvem do Google |
| **Sem Instalação** | Pronto no navegador |
| **Foco Nacional** | **IBGE**, **DataSUS** e fontes oficiais BR |
| **Memória persistente** | Achados salvos automaticamente no seu Google Drive |
| **Base Tecnológica** | **OpenCode** |

---

### 🚀 Como Usar

1. **Ambiente de execução → Executar tudo** (`Ctrl+F9`).
2. Aguarde ~2 min e clique em **ABRIR O UFVAI**.
3. **O keep-alive fica ativo em background** (1 linha a cada 60s em `ufvai_keepalive.log`) — impede o encerramento por ociosidade com a aba minimizada. Limite honesto: o Colab pode destruir o runtime (~12 h de teto, cota). Se cair: **Runtime → Reiniciar sessão → Executar tudo**.
4. **Para encerrar por completo** (economiza cota): clique no menu **Ambiente de execução → Desconectar e ecluir o ambiente de execução**.

> **Primeira vez:** tela de Termos **v2.2** (`terms_version=6`) — informe **nome + e-mail** para ativação (LGPD art. 7º, V, com registro de IP). Estatísticas anônimas ficam ativas por padrão, sem cookies, e podem ser desmarcadas (opt-out).
> **Depois:** "Bem-vindo de volta" com nome persistido — clique em *Continuar*.

---

- **Drive:** `Meu Drive/PesquisAI` · **Navegador:** Chrome · **Versão:** v0.7.11
- **Registro:** SisPPG/UFV nº 10356285004 · **Site oficial:** https://der.ufv.br/ufvai/


In [ ]:
import os, sys, shutil, socket, subprocess
from IPython.display import display, HTML, update_display

REPO = "https://github.com/gustavobraga-byte/versaoteste.git"
DIR = "/tmp/pesquisai"
PID = "ufvai_boot_panel"
CSS = "<style>.p{max-width:520px;margin:8px auto;padding:20px 24px;background:#f6f5f0;border:1px solid rgba(43,45,58,.14);border-radius:16px;font-family:system-ui,sans-serif}.w{font-size:30px;font-weight:800;text-align:center;color:#2b2d3a}.w b{color:#b8912f}.t{font-size:10px;letter-spacing:.35em;text-align:center;color:#8f8d86;margin-top:4px}.b{height:10px;background:#e9e6df;border-radius:999px;overflow:hidden;margin:14px 0 6px}.f{height:100%;width:PCT%;background:linear-gradient(90deg,#caa53f,#8f7536);border-radius:999px;transition:width .45s}.n{text-align:right;font-size:11px;color:#8f7536}.m{font-size:13px;color:#3c3f4e;margin-top:8px}</style>"

def show(p, m):
    return HTML(CSS.replace("PCT", str(p)) + f'<div class="p"><div class="w">UFV<b>AI</b></div><div class="t">INTELIGÊNCIA ARTIFICIAL</div><div class="b"><div class="f"></div></div><div class="n">{p}%</div><div class="m">{m}</div></div>')

def git(*a):
    try:
        return subprocess.run(["git", *a], capture_output=True, text=True, timeout=60).stdout.strip()
    except Exception:
        return ""

def ok(p):
    return (git("-C", p, "remote", "get-url", "origin").rstrip("/").removesuffix(".git") == REPO.removesuffix(".git")
            and os.path.isfile(os.path.join(p, "pesquisai", "ocweb.py")))

def porta(p):
    with socket.socket() as s:
        s.settimeout(.3)
        return s.connect_ex(("127.0.0.1", p)) == 0

def autopilot():
    return subprocess.run([sys.executable, "-c", "from pesquisai.obsidian.autopilot import recall, save"],
                          capture_output=True, cwd="/").returncode == 0

display(show(5, "Baixando o UFVAI..."), display_id=PID)
if ok(DIR):
    git("-C", DIR, "pull", "--ff-only", "--depth", "1")
else:
    shutil.rmtree(DIR, ignore_errors=True)
    r = subprocess.run(["git", "clone", "--depth", "1", REPO, DIR], capture_output=True, text=True)
    if r.returncode:
        raise SystemExit(f"[ERRO] git clone falhou:\n{r.stderr[-800:]}")
    if not ok(DIR):
        raise SystemExit("[ERRO] clone sem ocweb.py ou de outro repositório")

update_display(show(20, "Instalando o pacote pesquisai..."), display_id=PID)
for cmd in ([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", DIR],
            [sys.executable, "-m", "pip", "install", "-q", "-e", DIR],
            [sys.executable, "-m", "pip", "install", "-q", "--user", "--break-system-packages", "-e", DIR]):
    try:
        subprocess.run(cmd, capture_output=True, text=True, timeout=900)
    except Exception:
        pass
    if autopilot():
        break

os.chdir(DIR)
sys.path.insert(0, DIR)
if porta(8001):
    raise SystemExit("\n⚠️ Porta 8001 em uso (interface antiga).\nRuntime → Reiniciar sessão e rode de novo.\n")

update_display(show(60, "Iniciando o UFVAI..."), display_id=PID)
from main import run
run()

update_display(show(90, "Ativando o keep-alive..."), display_id=PID)
k = subprocess.Popen([sys.executable, "-u", "-c", "import time,datetime\nwhile True:\n    print('keepalive',datetime.datetime.now(),flush=True)\n    time.sleep(60)\n"],
                     cwd=DIR, stdin=subprocess.DEVNULL, stdout=open(os.path.join(DIR, "ufvai_keepalive.log"), "ab"),
                     stderr=subprocess.STDOUT, start_new_session=True)
open(os.path.join(DIR, "ufvai_keepalive.pid"), "w").write(str(k.pid))
update_display(show(100, f"UFVAI carregado. Keep-alive PID {k.pid}."), display_id=PID)


# ---

**Citação ABNT:**

```
BRAGA, Gustavo Bastos. UFVAI: agente de inteligência artificial para
pesquisa científica. Versão 0.7.11. Viçosa: Universidade Federal de Viçosa,
2026. Disponível em: https://colab.research.google.com/github/
gustavobraga-byte/versaoteste/. Projeto SisPPG/UFV nº 10356285004.
```

👨‍💻 **Gustavo Bastos Braga** (UFV) — [gustavo.braga@ufv.br](mailto:gustavo.braga@ufv.br)  
⭐ Código: [github.com/gustavobraga-byte/versaoteste](https://github.com/gustavobraga-byte/versaoteste)  
📜 MIT · [Termos v3](https://github.com/gustavobraga-byte/versaoteste/blob/main/docs/TERMS_OF_USE.md) · [Privacidade v1.1](https://github.com/gustavobraga-byte/versaoteste/blob/main/PRIVACY.md) · [Telemetria](https://github.com/gustavobraga-byte/versaoteste/blob/main/TELEMETRY.md) · [🌐 Site oficial](https://der.ufv.br/ufvai/)
